In [1]:
import pymongo
from bson import json_util

# ============================================================
# Everything held on one company: register record + raw filing,
# read straight from the source collections. Prints a summary,
# then the untouched documents.
# ============================================================
ORGNR = "990888213"          # organisasjonsnummer as a string, no spaces
SHOW_RAW = True              # False = summary only

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]

pipeline = [
    {"$match": {"organisasjonsnummer": ORGNR}},   # uses index organisasjonsnummer_1
    {"$lookup": {
        "from": "financial_data",
        "localField": "organisasjonsnummer",
        "foreignField": "_id",                    # _id IS the orgnr on that side
        "as": "financial",
    }},
    {"$addFields": {"financial": {"$first": "$financial"}}},   # 0 or 1 match
]

doc = next(db["companies"].aggregate(pipeline), None)


def dig(obj, path, default=None):
    """Walk a dotted path. Brreg omits fields rather than nulling them."""
    cur = obj
    for key in path.split("."):
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur if cur is not None else default


def money(v, unit):
    return "-".rjust(18) if v is None else f"{v:,.0f} {unit}".rjust(18)


def line(label, value):
    print(f"  {label:<34}{value}")


# ------------------------------------------------------------
# Not in the register snapshot at all
# ------------------------------------------------------------
if doc is None:
    fin = db["financial_data"].find_one({"_id": ORGNR})
    print(f"No document in companies for {ORGNR}.")
    print("financial_data:", "present" if fin else "absent")
    if fin and SHOW_RAW:
        print(json_util.dumps(fin, indent=2, ensure_ascii=False))
    raise SystemExit

# ------------------------------------------------------------
# Identity
# ------------------------------------------------------------
print("=" * 72)
print(f"{doc.get('navn', '(no name)')}")
print(f"orgnr {doc['organisasjonsnummer']}   "
      f"{dig(doc, 'organisasjonsform.kode', '?')} - {dig(doc, 'organisasjonsform.beskrivelse', '')}")
print("=" * 72)

print("\nREGISTER")
line("Registered (Enhetsregisteret)", dig(doc, "registreringsdatoEnhetsregisteret", "-"))
line("Founded (stiftelsesdato)", dig(doc, "stiftelsesdato", "-"))
line("Industry (NACE 1)", f"{dig(doc, 'naeringskode1.kode', '-')}  "
                          f"{dig(doc, 'naeringskode1.beskrivelse', '')}")
line("Sector", f"{dig(doc, 'institusjonellSektorkode.kode', '-')}  "
                f"{dig(doc, 'institusjonellSektorkode.beskrivelse', '')}")
line("Employees", dig(doc, "antallAnsatte", "not registered"))
line("Last filing per register", dig(doc, "sisteInnsendteAarsregnskap", "-"))
flags = [name for name, key in (
    ("bankrupt", "konkurs"),
    ("under liquidation", "underAvvikling"),
    ("compulsory liquidation", "underTvangsavviklingEllerTvangsopplosning"),
) if doc.get(key)]
line("Status flags", ", ".join(flags) if flags else "none set")

# ------------------------------------------------------------
# Location - forretningsadresse is the operating address,
# postadresse the mailing one. Either may be absent.
# ------------------------------------------------------------
print("\nLOCATION")
for label, key in (("Business address", "forretningsadresse"),
                   ("Postal address", "postadresse")):
    addr = doc.get(key)
    if not addr:
        line(label, "-")
        continue
    street = ", ".join(addr.get("adresse") or []) or "-"
    line(label, street)
    line("", f"{dig(addr, 'postnummer', '')} {dig(addr, 'poststed', '')}".strip())
    line("", f"kommune {dig(addr, 'kommune', '-')} "
             f"({dig(addr, 'kommunenummer', '-')}), {dig(addr, 'land', '-')}")

# ------------------------------------------------------------
# Financial statements
# ------------------------------------------------------------
fin = doc.get("financial")
print("\nFINANCIAL DATA")

if fin is None:
    print("  No document. Never definitively answered - never attempted, added by a")
    print("  later register snapshot, or last attempt was HTTP 500/429/network error.")
    print("  Eligible for retry on the next fetch run.")
elif fin.get("fetch_status") == "no_data":
    line("Status", f"no_data (HTTP 404) as of {fin.get('fetched_at')}")
    print("  Confirmed: nothing filed as of that date. Not necessarily permanent.")
else:
    filings = fin.get("data") or []
    line("Status", f"{fin.get('fetch_status')} (HTTP {fin.get('http_status')}), "
                   f"fetched {fin.get('fetched_at')}")
    line("Filings returned", len(filings))

    for i, f in enumerate(filings, 1):
        cur = dig(f, "valuta", "?")
        rev = dig(f, "resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter")
        opres = dig(f, "resultatregnskapResultat.driftsresultat.driftsresultat")
        assets = dig(f, "eiendeler.sumEiendeler")
        eqliab = dig(f, "egenkapitalGjeld.sumEgenkapitalGjeld")

        print(f"\n  --- Filing {i} of {len(filings)} "
              f"({dig(f, 'regnskapsperiode.fraDato', '?')} to "
              f"{dig(f, 'regnskapsperiode.tilDato', '?')}) ---")
        line("Currency", cur)
        line("Statement type", f"{dig(f, 'regnskapstype', '-')} / "
                               f"{dig(f, 'oppstillingsplan', '-')}")
        line("Small entity rules", dig(f, "regnkapsprinsipper.smaaForetak", "-"))
        line("Unaudited", dig(f, "revisjon.ikkeRevidertAarsregnskap", "-"))
        line("Liquidation accounts", dig(f, "avviklingsregnskap", "-"))
        line("Parent company", dig(f, "virksomhet.morselskap", "-"))

        print("\n  Income statement")
        line("Operating revenue", money(rev, cur))
        line("Operating costs", money(dig(f, "resultatregnskapResultat.driftsresultat"
                                             ".driftskostnad.sumDriftskostnad"), cur))
        line("Operating result", money(opres, cur))
        line("Net financial items", money(dig(f, "resultatregnskapResultat"
                                                 ".finansresultat.nettoFinans"), cur))
        line("Pre-tax result", money(dig(f, "resultatregnskapResultat"
                                            ".ordinaertResultatFoerSkattekostnad"), cur))
        line("Net result", money(dig(f, "resultatregnskapResultat.aarsresultat"), cur))

        print("\n  Balance sheet")
        line("Fixed assets", money(dig(f, "eiendeler.anleggsmidler.sumAnleggsmidler"), cur))
        line("Current assets", money(dig(f, "eiendeler.omloepsmidler.sumOmloepsmidler"), cur))
        line("Total assets", money(assets, cur))
        line("Equity", money(dig(f, "egenkapitalGjeld.egenkapital.sumEgenkapital"), cur))
        line("Total liabilities", money(dig(f, "egenkapitalGjeld.gjeldOversikt.sumGjeld"), cur))
        line("Equity + liabilities", money(eqliab, cur))

        print("\n  Derived")
        # Guard both the missing case and the zero-revenue case: a third of
        # filings report no operating revenue, so the ratio is undefined there.
        if rev is None or opres is None:
            line("Operating margin", "undefined (missing figure)")
        elif rev == 0:
            line("Operating margin", "undefined (zero revenue)")
        else:
            line("Operating margin", f"{opres / rev * 100:>17.1f} %")

        if assets is None or eqliab is None:
            line("Balance identity", "cannot check (missing figure)")
        else:
            diff = assets - eqliab
            line("Balance identity",
                 "balances" if abs(diff) <= 1.5 else f"OFF BY {diff:,.0f} {cur}")

        if cur != "NOK":
            print(f"\n  NOTE: reported in {cur}, not comparable with NOK figures unconverted.")

# ------------------------------------------------------------
# Raw documents, unmodified
# ------------------------------------------------------------
if SHOW_RAW:
    print("\n" + "=" * 72)
    print("RAW DOCUMENTS")
    print("=" * 72)
    print(json_util.dumps(doc, indent=2, ensure_ascii=False))

EQUINOR ENERGY AS
orgnr 990888213   AS - Aksjeselskap

REGISTER
  Registered (Enhetsregisteret)     2007-02-15
  Founded (stiftelsesdato)          2007-02-01
  Industry (NACE 1)                 06.200  Utvinning av naturgass
  Sector                            1120  Statlig eide aksjeselskaper mv.
  Employees                         not registered
  Last filing per register          2025
  Status flags                      none set

LOCATION
  Business address                  Forusbeen 50
                                    4035 STAVANGER
                                    kommune STAVANGER (1103), Norge
  Postal address                    Postboks 8500 Forus
                                    4035 STAVANGER
                                    kommune STAVANGER (1103), Norge

FINANCIAL DATA
  Status                            success (HTTP 200), fetched 2026-08-27 16:47:06.361000
  Filings returned                  1

  --- Filing 1 of 1 (2025-01-01 to 2025-12-31) ---
  Currency    

In [5]:
import os

import pandas as pd
from pyspark.sql import SparkSession, functions as F

# ============================================================
# Company registrations by NACE section and year, as percentages.
# Five denominators, so the effect of each non-section bucket is
# visible rather than assumed.
# Reads the Parquet export of `companies`: 2 columns out of 63.
# ============================================================
YEAR_FROM = 1996
YEAR_TO = 2025

# Which date defines the cohort.
#   "registreringsdatoEnhetsregisteret" - entry into Enhetsregisteret. The
#       register opened in 1995, so years up to and including 1995 are a
#       backlog load of already-existing entities, not new formations.
#   "stiftelsesdato"                    - legal founding date. Can predate
#       registration by decades and is absent for many legal forms.
DATE_FIELD = "registreringsdatoEnhetsregisteret"

LEGAL_FORMS = None          # None = whole register; {"AS"} = aksjeselskap only

DATA_DIR = "/home/jovyan/data"
COMPANIES_PARQUET = os.path.join(DATA_DIR, "parquet", "companies")
OUT_PREFIX = os.path.join(DATA_DIR, "sector_by_year")

# ------------------------------------------------------------
# SN2025 division -> section, the standard in force in Enhetsregisteret
# since 2025-09-01. SN2025 is the Norwegian implementation of NACE Rev. 2.1.
# 87 divisions across 22 sections. Division 45 does not exist (its activities
# went to 46, 47 and 95); division 98 does.
# Verified against SSB KLASS classification 6, codes valid 2026-01-01.
# ------------------------------------------------------------
SECTION_RANGES = [
    ("A",  1,  3, "Agriculture, forestry and fishing"),
    ("B",  5,  9, "Mining and quarrying"),
    ("C", 10, 33, "Manufacturing"),
    ("D", 35, 35, "Electricity, gas, steam, air conditioning"),
    ("E", 36, 39, "Water supply, sewerage, waste management"),
    ("F", 41, 43, "Construction"),
    ("G", 46, 47, "Wholesale and retail trade"),
    ("H", 49, 53, "Transportation and storage"),
    ("I", 55, 56, "Accommodation and food service"),
    ("J", 58, 60, "Publishing, broadcasting, content production"),
    ("K", 61, 63, "Telecoms, IT and information services"),
    ("L", 64, 66, "Financial and insurance activities"),
    ("M", 68, 68, "Real estate activities"),
    ("N", 69, 75, "Professional, scientific and technical"),
    ("O", 77, 82, "Administrative and support service"),
    ("P", 84, 84, "Public administration and defence"),
    ("Q", 85, 85, "Education"),
    ("R", 86, 88, "Human health and social work"),
    ("S", 90, 93, "Arts, entertainment and recreation"),
    ("T", 94, 96, "Other service activities, incl. vehicle repair"),
    ("U", 97, 98, "Households as employers"),
    ("V", 99, 99, "Extraterritorial organisations"),
]
SECTIONS = [letter for letter, _, _, _ in SECTION_RANGES]

UNSPEC = "00"           # division 00 - the register's own "unspecified"
NO_CODE = "--"          # naeringskode1 absent entirely
UNMAPPED = "??"         # code string present but placed in no section
YEARS = range(YEAR_FROM, YEAR_TO + 1)

# Division 45 exists only in SN2007. Any hit means this mapping is the wrong
# vintage for the data and every letter from J onwards is suspect.
SN2007_ONLY_DIVISION = 45


def section_expr(code, division):
    """
    Map a NACE code to a section letter.

    Order matters:
      NO_CODE   naeringskode1.kode is null - the register says nothing
      UNMAPPED  the string is present but its division will not parse
      UNSPEC    division 00 - a deliberate register value, not an error
      A-V       a real SN2025 section
      UNMAPPED  anything else, including SN2007 leftovers like division 45
    """
    expr = F.lit(UNMAPPED)
    for letter, lo, hi, _ in reversed(SECTION_RANGES):
        expr = F.when(division.between(lo, hi), F.lit(letter)).otherwise(expr)
    return (F.when(code.isNull(), F.lit(NO_CODE))
             .when(division.isNull(), F.lit(UNMAPPED))
             .when(division == 0, F.lit(UNSPEC))
             .otherwise(expr))


spark = SparkSession.builder.appName("group13_sector_by_year").getOrCreate()

df = spark.read.parquet(COMPANIES_PARQUET)
if LEGAL_FORMS is not None:
    df = df.filter(F.col("organisasjonsform.kode").isin(list(LEGAL_FORMS)))

# Dates are strings, "YYYY-MM-DD". try_cast (Spark 4) yields null on a
# malformed value instead of failing the job under ANSI mode.
year = F.substring(F.col(DATE_FIELD), 1, 4).try_cast("int")

# naeringskode1.kode looks like "68.200"; the division is the first two digits.
code = F.col("naeringskode1.kode")
division = F.substring(code, 1, 2).try_cast("int")

work = df.select(
    year.alias("year"),
    division.alias("division"),
    section_expr(code, division).alias("section"),
    code.alias("nace"),
).cache()

# ------------------------------------------------------------
# Vintage guard, before anything is tabulated
# ------------------------------------------------------------
sn2007_hits = work.filter(F.col("division") == SN2007_ONLY_DIVISION).count()
if sn2007_hits:
    print("*" * 72)
    print("WARNING: %d entities carry division %02d, which exists only in SN2007."
          % (sn2007_hits, SN2007_ONLY_DIVISION))
    print("The section letters below assume SN2025. Every letter from J onwards")
    print("is wrong for those rows. Stop and resolve the vintage before using")
    print("these tables.")
    print("*" * 72)
else:
    print("Vintage guard: no division %02d present, consistent with SN2025.\n"
          % SN2007_ONLY_DIVISION)

# ------------------------------------------------------------
# Coverage, measured rather than assumed
# ------------------------------------------------------------
total_rows = work.count()
no_year = work.filter(F.col("year").isNull()).count()
in_window = work.filter(F.col("year").between(YEAR_FROM, YEAR_TO))
window_rows = in_window.count()

print("Source:            %s" % COMPANIES_PARQUET)
print("Cohort date field: %s" % DATE_FIELD)
print("Legal forms:       %s" % (sorted(LEGAL_FORMS) if LEGAL_FORMS else "ALL"))
print()
print("Rows in register:        %9d" % total_rows)
print("  no date value:         %9d" % no_year)
print("  outside %d-%d:      %9d" % (YEAR_FROM, YEAR_TO,
                                     total_rows - window_rows - no_year))
print("  in window, tabulated:  %9d" % window_rows)

# The years around the window start, so the size of the excluded mass is
# visible rather than asserted. Enhetsregisteret's opening year should dominate.
edge = (work.filter(F.col("year").between(YEAR_FROM - 4, YEAR_FROM))
        .groupBy("year").count().orderBy("year").toPandas())
print("\nRegistrations just before and at the window start:")
for _, r in edge.iterrows():
    print("  %d  %9d" % (r["year"], r["count"]))

# What landed in the unmapped bucket. A null `nace` means the string was
# present but unparseable; anything else is a code outside SN2025.
unmapped = (in_window.filter(F.col("section") == UNMAPPED)
            .groupBy("nace").count().orderBy(F.desc("count")).limit(10).toPandas())
if len(unmapped):
    print("\nCodes placed in '%s' (top 10):" % UNMAPPED)
    for _, r in unmapped.iterrows():
        label = "(unparseable)" if pd.isna(r["nace"]) else r["nace"]
        print("  %-14s %9d" % (label, r["count"]))
else:
    print("\nEvery naeringskode1 present maps to an SN2025 section or to '%s'." % UNSPEC)

# ------------------------------------------------------------
# The aggregation: 1.17M rows down to at most 30 x 25 cells
# ------------------------------------------------------------
counts = in_window.groupBy("year", "section").count().toPandas()

work.unpersist()
spark.stop()

# ------------------------------------------------------------
# Pivot once, then renormalise per denominator
# ------------------------------------------------------------
ALL_COLUMNS = SECTIONS + [UNSPEC, NO_CODE, UNMAPPED]
table = (counts.pivot(index="year", columns="section", values="count")
         .reindex(index=YEARS, columns=ALL_COLUMNS)
         .fillna(0).astype(int))

print("\nSN2025 sections")
for letter, lo, hi, desc in SECTION_RANGES:
    span = "%02d" % lo if lo == hi else "%02d-%02d" % (lo, hi)
    print("  %s  %-6s %s" % (letter, span, desc))
print("  %s      division 00, unspecified activity" % UNSPEC)
print("  %s      naeringskode1 absent" % NO_CODE)
print("  %s      code present, placed in no section" % UNMAPPED)

# Each variant selects its own columns; the denominator is the row sum of
# exactly those columns, so every table's rows sum to 100.
VARIANTS = [
    ("pct_all", SECTIONS + [UNSPEC, NO_CODE, UNMAPPED],
     "TABLE 1  All registrations (every bucket included)"),
    ("pct_sections", SECTIONS,
     "TABLE 2  Sections only (all three buckets excluded)"),
    ("pct_nocode", SECTIONS + [NO_CODE],
     "TABLE 3  Sections plus '%s' (no code), excluding '%s' and '%s'"
     % (NO_CODE, UNSPEC, UNMAPPED)),
    ("pct_unmapped", SECTIONS + [UNMAPPED],
     "TABLE 4  Sections plus '%s' (unmappable), excluding '%s' and '%s'"
     % (UNMAPPED, UNSPEC, NO_CODE)),
    ("pct_unspec", SECTIONS + [UNSPEC],
     "TABLE 5  Sections plus '%s' (unspecified), excluding '%s' and '%s'"
     % (UNSPEC, NO_CODE, UNMAPPED)),
]

for suffix, cols, title in VARIANTS:
    subset = table[cols]
    totals = subset.sum(axis=1)
    # replace(0, NA) so a year with no rows gives NaN, not a divide error
    pct = subset.div(totals.replace(0, pd.NA), axis=0) * 100

    print("\n" + title)
    header = "Year " + "".join("%7s" % c for c in cols) + "%12s" % "Total"
    print(header)
    print("-" * len(header))
    for y in YEARS:
        if totals[y] == 0:
            print("%-5d%s%12d" % (y, "".join("%7s" % "-" for _ in cols), 0))
            continue
        cells = "".join("%7.2f" % pct.loc[y, c] for c in cols)
        print("%-5d%s%12d" % (y, cells, totals[y]))

    # The total column is named "total" rather than "N", which would collide
    # with NACE section N.
    out = pct.round(2)
    out["total"] = totals
    path = "%s_%s.csv" % (OUT_PREFIX, suffix)
    out.to_csv(path, index_label="year")
    print("Written: %s" % path)

Vintage guard: no division 45 present, consistent with SN2025.

Source:            /home/jovyan/data/parquet/companies
Cohort date field: registreringsdatoEnhetsregisteret
Legal forms:       ALL

Rows in register:          1171373
  no date value:                 0
  outside 1996-2025:         152799
  in window, tabulated:    1018574

Registrations just before and at the window start:
  1995      91047
  1996      12095

Every naeringskode1 present maps to an SN2025 section or to '00'.

SN2025 sections
  A  01-03  Agriculture, forestry and fishing
  B  05-09  Mining and quarrying
  C  10-33  Manufacturing
  D  35     Electricity, gas, steam, air conditioning
  E  36-39  Water supply, sewerage, waste management
  F  41-43  Construction
  G  46-47  Wholesale and retail trade
  H  49-53  Transportation and storage
  I  55-56  Accommodation and food service
  J  58-60  Publishing, broadcasting, content production
  K  61-63  Telecoms, IT and information services
  L  64-66  Financial and 

In [6]:
import os

import pandas as pd
from pyspark.sql import SparkSession, functions as F

# ============================================================
# Is division 00 a coding backlog, or something structural?
# Reads the Parquet export of `companies`.
# ============================================================
DATA_DIR = "/home/jovyan/data"
COMPANIES_PARQUET = os.path.join(DATA_DIR, "parquet", "companies")

DATE_FIELD = "registreringsdatoEnhetsregisteret"

# A cohort old enough that any short processing lag has expired, but recent
# enough to sit inside the elevated-00 era. Used for the signal cuts.
COHORT_FROM, COHORT_TO = 2019, 2023

spark = SparkSession.builder.appName("group13_probe_unspecified").getOrCreate()

code = F.col("naeringskode1.kode")
division = F.substring(code, 1, 2).try_cast("int")

df = spark.read.parquet(COMPANIES_PARQUET).select(
    F.substring(F.col(DATE_FIELD), 1, 4).try_cast("int").alias("year"),
    F.substring(F.col(DATE_FIELD), 1, 7).alias("month"),
    F.col("organisasjonsform.kode").alias("form"),
    # 1 when the entity carries the unspecified code, 0 when it carries a real
    # one. Entities with no naeringskode1 at all are excluded below, so the
    # two buckets are not conflated.
    F.when(division == 0, F.lit(1)).otherwise(F.lit(0)).alias("unspec"),
    code.isNotNull().alias("has_code"),
    F.coalesce(F.col("registrertIForetaksregisteret"), F.lit(False)).alias("in_foretaksreg"),
    F.coalesce(F.col("registrertIMvaregisteret"), F.lit(False)).alias("in_mva"),
    F.coalesce(F.col("antallAnsatte"), F.lit(0)).alias("employees"),
    F.col("sisteInnsendteAarsregnskap").isNotNull().alias("has_filing"),
    F.coalesce(F.col("konkurs"), F.lit(False)).alias("bankrupt"),
    F.coalesce(F.col("underAvvikling"), F.lit(False)).alias("liquidating"),
).filter(F.col("has_code")).cache()   # entities with no code at all are a separate question

print("Entities with a naeringskode1: %d\n" % df.count())


def share_table(grouped):
    """Share of rows carrying division 00, per group."""
    p = grouped.toPandas()
    p["pct_00"] = p["n_unspec"] / p["n"] * 100
    return p


# ------------------------------------------------------------
# A. Month of registration. The lag test.
#    A processing backlog should show 00 rising steeply with recency and
#    peaking in the months just before the snapshot (2026-08-25).
# ------------------------------------------------------------
monthly = share_table(
    df.filter(F.col("year") >= 2023)
      .groupBy("month")
      .agg(F.count("*").alias("n"), F.sum("unspec").alias("n_unspec"))
      .orderBy("month")
)
print("A. Share of division 00 by registration month, 2023 onwards")
print("   Month      n      00      %")
print("   " + "-" * 34)
for _, r in monthly.iterrows():
    print("   %s %7d %7d %6.2f" % (r["month"], r["n"], r["n_unspec"], r["pct_00"]))

# ------------------------------------------------------------
# B. Legal form, old era against new. A backlog should raise 00 across all
#    forms roughly evenly; a structural cause should concentrate.
# ------------------------------------------------------------
by_form = share_table(
    df.filter(F.col("year").between(1996, 2025))
      .withColumn("era", F.when(F.col("year") <= 2017, F.lit("1996-2017"))
                          .otherwise(F.lit("2018-2025")))
      .groupBy("form", "era")
      .agg(F.count("*").alias("n"), F.sum("unspec").alias("n_unspec"))
)
pivot_n = by_form.pivot(index="form", columns="era", values="n").fillna(0)
pivot_p = by_form.pivot(index="form", columns="era", values="pct_00")
top_forms = pivot_n.sum(axis=1).sort_values(ascending=False).head(12).index

print("\nB. Share of division 00 by legal form and era")
print("   Form        n 96-17   %% 96-17     n 18-25   %% 18-25")
print("   " + "-" * 52)
for form in top_forms:
    print("   %-8s %9.0f %9s   %9.0f %9s" % (
        form,
        pivot_n.loc[form].get("1996-2017", 0),
        "%.2f" % pivot_p.loc[form]["1996-2017"] if pd.notna(pivot_p.loc[form].get("1996-2017")) else "-",
        pivot_n.loc[form].get("2018-2025", 0),
        "%.2f" % pivot_p.loc[form]["2018-2025"] if pd.notna(pivot_p.loc[form].get("2018-2025")) else "-",
    ))

# ------------------------------------------------------------
# C. Activity and status signals, within one cohort.
#    A dormant shell with nothing to classify looks different from a coded
#    entity awaiting processing.
# ------------------------------------------------------------
cohort = df.filter(F.col("year").between(COHORT_FROM, COHORT_TO))
cohort_n = cohort.count()
print("\nC. Signal cuts within the %d-%d cohort (n = %d)"
      % (COHORT_FROM, COHORT_TO, cohort_n))
print("   Signal                      n true    %% 00 true     n false   %% 00 false")
print("   " + "-" * 72)

SIGNALS = [
    ("In Foretaksregisteret", F.col("in_foretaksreg")),
    ("In MVA register", F.col("in_mva")),
    ("Has employees", F.col("employees") > 0),
    ("Has a reported filing", F.col("has_filing")),
    ("Bankrupt", F.col("bankrupt")),
    ("Under liquidation", F.col("liquidating")),
]

for label, condition in SIGNALS:
    row = cohort.select(
        F.sum(F.when(condition, 1).otherwise(0)).alias("n_true"),
        F.sum(F.when(condition, F.col("unspec")).otherwise(0)).alias("u_true"),
        F.sum(F.when(condition, 0).otherwise(1)).alias("n_false"),
        F.sum(F.when(condition, 0).otherwise(F.col("unspec"))).alias("u_false"),
    ).collect()[0]
    pt = row["u_true"] / row["n_true"] * 100 if row["n_true"] else float("nan")
    pf = row["u_false"] / row["n_false"] * 100 if row["n_false"] else float("nan")
    print("   %-24s %9d %11.2f   %9d %11.2f"
          % (label, row["n_true"], pt, row["n_false"], pf))

# ------------------------------------------------------------
# D. Does 00 sit on a handful of five-digit codes, or exactly one?
#    SN2025 has a single unspecified code; anything else here is a surprise.
# ------------------------------------------------------------
codes = (spark.read.parquet(COMPANIES_PARQUET)
         .select(F.col("naeringskode1.kode").alias("nace"))
         .filter(F.col("nace").startswith("00"))
         .groupBy("nace").count().orderBy(F.desc("count")).toPandas())
print("\nD. Five-digit codes inside division 00")
for _, r in codes.iterrows():
    print("   %-10s %9d" % (r["nace"], r["count"]))

df.unpersist()
spark.stop()

Entities with a naeringskode1: 1135646

A. Share of division 00 by registration month, 2023 onwards
   Month      n      00      %
   ----------------------------------
   2023-01    6327     588   9.29
   2023-02    6816     493   7.23
   2023-03    7644     595   7.78
   2023-04    4002     305   7.62
   2023-05    4780     407   8.51
   2023-06    5649     409   7.24
   2023-07    5363     423   7.89
   2023-08    5805     457   7.87
   2023-09    5103     392   7.68
   2023-10    7269     601   8.27
   2023-11    5392     549  10.18
   2023-12    4534     745  16.43
   2024-01    6955     829  11.92
   2024-02    5923     465   7.85
   2024-03    4325     394   9.11
   2024-04    5955     503   8.45
   2024-05    4979     417   8.38
   2024-06    6013     537   8.93
   2024-07    5724     503   8.79
   2024-08    4796     389   8.11
   2024-09    6200     584   9.42
   2024-10    7536     669   8.88
   2024-11    5891     764  12.97
   2024-12    4915     693  14.10
   2025-01    6

In [7]:
import os

from pyspark.sql import SparkSession, functions as F

# ============================================================
# Division 00 within AS: is it substitution for 64/68, and are
# 00-coded AS dormant?
# ============================================================
DATA_DIR = "/home/jovyan/data"
COMPANIES_PARQUET = os.path.join(DATA_DIR, "parquet", "companies")
DATE_FIELD = "registreringsdatoEnhetsregisteret"
YEAR_FROM, YEAR_TO = 2010, 2025

spark = SparkSession.builder.appName("group13_probe_unspecified_as").getOrCreate()

code = F.col("naeringskode1.kode")
division = F.substring(code, 1, 2).try_cast("int")

# AS only, so nothing below can be explained by legal form.
a_s = (spark.read.parquet(COMPANIES_PARQUET)
       .filter(F.col("organisasjonsform.kode") == "AS")
       .filter(code.isNotNull())
       .select(
           F.substring(F.col(DATE_FIELD), 1, 4).try_cast("int").alias("year"),
           division.alias("div"),
           F.coalesce(F.col("registrertIMvaregisteret"), F.lit(False)).alias("in_mva"),
           F.coalesce(F.col("antallAnsatte"), F.lit(0)).alias("employees"),
           F.col("sisteInnsendteAarsregnskap").isNotNull().alias("has_filing"),
           F.coalesce(F.col("erIKonsern"), F.lit(False)).alias("in_group"),
           F.col("overordnetEnhet").isNotNull().alias("has_parent"),
           F.coalesce(F.col("underAvvikling"), F.lit(False)).alias("liquidating"),
           F.col("kapital.belop").alias("capital"),
       )
       .filter(F.col("year").between(YEAR_FROM, YEAR_TO))
       .cache())

print("AS with a naeringskode1, %d-%d: %d\n" % (YEAR_FROM, YEAR_TO, a_s.count()))

# ------------------------------------------------------------
# A. Counts per year for the divisions that matter, AS only.
#    Substitution shows as 64 and 68 flattening while 00 climbs.
# ------------------------------------------------------------
WATCH = [0, 64, 68, 70]     # unspecified, financial, real estate, head offices
per_year = (a_s.withColumn("bucket",
                           F.when(F.col("div").isin(WATCH), F.col("div"))
                            .otherwise(F.lit(-1)))
            .groupBy("year", "bucket").count().toPandas()
            .pivot(index="year", columns="bucket", values="count")
            .fillna(0).astype(int))

print("A. AS registrations by division and year (count, then %% of AS that year)")
header = "Year" + "".join("%10s" % ("div %02d" % d) for d in WATCH) + "%10s%10s" % ("other", "AS total")
print(header)
print("-" * len(header))
for y in range(YEAR_FROM, YEAR_TO + 1):
    row = per_year.loc[y]
    total = row.sum()
    cells = "".join("%10d" % row.get(d, 0) for d in WATCH)
    print("%-4d%s%10d%10d" % (y, cells, row.get(-1, 0), total))
print()
for y in range(YEAR_FROM, YEAR_TO + 1):
    row = per_year.loc[y]
    total = row.sum()
    cells = "".join("%10.2f" % (row.get(d, 0) / total * 100) for d in WATCH)
    print("%-4d%s%10.2f%10d" % (y, cells, row.get(-1, 0) / total * 100, total))

# ------------------------------------------------------------
# B. Dormancy signals, AS only, one cohort. Form is now held constant,
#    so a gap here is about the entity rather than its legal shape.
# ------------------------------------------------------------
cohort = a_s.filter(F.col("year").between(2019, 2023))
n = cohort.count()
print("\nB. Signals within AS registered 2019-2023 (n = %d)" % n)
print("   Signal                    n true    %% 00 true      n false   %% 00 false")
print("   " + "-" * 70)

SIGNALS = [
    ("In MVA register", F.col("in_mva")),
    ("Has employees", F.col("employees") > 0),
    ("Has a reported filing", F.col("has_filing")),
    ("Part of a group", F.col("in_group")),
    ("Has a parent entity", F.col("has_parent")),
    ("Under liquidation", F.col("liquidating")),
    ("Capital above 30k", F.col("capital") > 30000),
]

unspec = F.when(F.col("div") == 0, 1).otherwise(0)
for label, cond in SIGNALS:
    r = cohort.select(
        F.sum(F.when(cond, 1).otherwise(0)).alias("nt"),
        F.sum(F.when(cond, unspec).otherwise(0)).alias("ut"),
        F.sum(F.when(cond, 0).otherwise(1)).alias("nf"),
        F.sum(F.when(cond, 0).otherwise(unspec)).alias("uf"),
    ).collect()[0]
    pt = r["ut"] / r["nt"] * 100 if r["nt"] else float("nan")
    pf = r["uf"] / r["nf"] * 100 if r["nf"] else float("nan")
    print("   %-22s %9d %11.2f    %9d %11.2f" % (label, r["nt"], pt, r["nf"], pf))

a_s.unpersist()
spark.stop()

AS with a naeringskode1, 2010-2025: 302593

A. AS registrations by division and year (count, then %% of AS that year)
Year    div 00    div 64    div 68    div 70     other  AS total
----------------------------------------------------------------
2010       174       339      1849       275      3583      6220
2011       195       415      2271       348      4462      7691
2012       316       516      2903       481      7546     11762
2013       361       798      3087       597      7841     12684
2014       388       761      3128       633      7974     12884
2015       494       972      3446       660      8446     14018
2016       613      1076      3865       735      9016     15305
2017       865      1432      4347       785     10222     17651
2018      1775      1228      4094       794     10897     18788
2019      3001       190      4873       861     11371     20296
2020      4474       220      4968       987     13142     23791
2021      6281       290      5998   

In [8]:
import os

import pandas as pd
from pyspark.sql import SparkSession, functions as F

# ============================================================
# What is in division 00?
#   Part A: profile 00 AS against non-00 AS, split by recency, to separate
#           a short coding queue from a settled population.
#   Part B: read vedtektsfestetFormaal to test the holding-company reading
#           directly rather than inferring it from dormancy proxies.
# Reads the Parquet export of `companies`.
# ============================================================
DATA_DIR = "/home/jovyan/data"
COMPANIES_PARQUET = os.path.join(DATA_DIR, "parquet", "companies")
DATE_FIELD = "registreringsdatoEnhetsregisteret"

# Settled: long enough after registration that any coding queue has drained.
# Recent: the months immediately before the 2026-08-25 snapshot.
SETTLED_FROM, SETTLED_TO = "2019-01", "2023-12"
RECENT_FROM, RECENT_TO = "2026-03", "2026-08"

spark = SparkSession.builder.appName("group13_probe_division_00").getOrCreate()

code = F.col("naeringskode1.kode")
division = F.substring(code, 1, 2).try_cast("int")

# vedtektsfestetFormaal is an array of strings. Flatten to one lowercase blob;
# coalesce first so a null array becomes an empty string rather than null.
formaal = F.lower(F.coalesce(
    F.array_join(F.col("vedtektsfestetFormaal"), " "), F.lit("")))

# AS only, so nothing below can be explained by legal form.
base = (spark.read.parquet(COMPANIES_PARQUET)
        .filter(F.col("organisasjonsform.kode") == "AS")
        .filter(code.isNotNull())
        .select(
            F.substring(F.col(DATE_FIELD), 1, 7).alias("month"),
            division.alias("div"),
            F.coalesce(F.col("registrertIMvaregisteret"), F.lit(False)).alias("in_mva"),
            F.coalesce(F.col("antallAnsatte"), F.lit(0)).alias("employees"),
            F.col("sisteInnsendteAarsregnskap").isNotNull().alias("has_filing"),
            F.coalesce(F.col("erIKonsern"), F.lit(False)).alias("in_group"),
            F.coalesce(F.col("underAvvikling"), F.lit(False)).alias("liquidating"),
            formaal.alias("formaal"),
        )
        .withColumn("cohort",
                    F.when(F.col("month").between(SETTLED_FROM, SETTLED_TO),
                           F.lit("settled"))
                     .when(F.col("month").between(RECENT_FROM, RECENT_TO),
                           F.lit("recent"))
                     .otherwise(F.lit(None)))
        .filter(F.col("cohort").isNotNull())
        .withColumn("is00", F.col("div") == 0)
        .cache())

sizes = base.groupBy("cohort", "is00").count().toPandas()
print("Cohorts (AS with a naeringskode1)")
print("  settled = %s to %s     recent = %s to %s"
      % (SETTLED_FROM, SETTLED_TO, RECENT_FROM, RECENT_TO))
for _, r in sizes.sort_values(["cohort", "is00"]).iterrows():
    print("  %-8s %-8s %8d" % (r["cohort"], "div 00" if r["is00"] else "other", r["count"]))

# ------------------------------------------------------------
# PART A. Profile within each group.
#   A settled population of passive shells should look the same in both
#   cohorts. A coding queue should make the recent 00 group look more like
#   ordinary operating companies, because it contains entities that will
#   later be recoded away from 00.
# ------------------------------------------------------------
SIGNALS = [
    ("VAT registered", F.col("in_mva")),
    ("Has employees", F.col("employees") > 0),
    ("Has a reported filing", F.col("has_filing")),
    ("Flagged in a group", F.col("in_group")),
    ("Under liquidation", F.col("liquidating")),
    ("Purpose text present", F.length(F.col("formaal")) > 0),
]

rows = []
for cohort in ("settled", "recent"):
    for is00 in (True, False):
        grp = base.filter((F.col("cohort") == cohort) & (F.col("is00") == is00))
        n = grp.count()
        agg = grp.select(*[
            F.sum(F.when(cond, 1).otherwise(0)).alias("s%d" % i)
            for i, (_, cond) in enumerate(SIGNALS)
        ]).collect()[0]
        rows.append({
            "cohort": cohort,
            "group": "div 00" if is00 else "other",
            "n": n,
            **{label: (agg["s%d" % i] / n * 100 if n else float("nan"))
               for i, (label, _) in enumerate(SIGNALS)},
        })

profile = pd.DataFrame(rows)
print("\nPART A. Profile by cohort and group (%% of that group)")
header = "%-9s%-8s%9s" % ("Cohort", "Group", "n") + "".join(
    "%24s" % label for label, _ in SIGNALS)
print(header)
print("-" * len(header))
for _, r in profile.iterrows():
    print("%-9s%-8s%9d" % (r["cohort"], r["group"], r["n"])
          + "".join("%24.2f" % r[label] for label, _ in SIGNALS))

print("\n  Read: if 'div 00 / recent' shows materially more VAT registration or")
print("  more employers than 'div 00 / settled', the excess is the coding queue.")
print("  If the two are alike, 00 is a settled population of passive entities.")

# ------------------------------------------------------------
# PART B. Statutory purpose text.
#   Keywords are counted independently first (an entity can match several),
#   then assigned to one bucket by the priority order below.
# ------------------------------------------------------------
KEYWORDS = [
    ("holding", ["holding", "holdingselskap"]),
    ("share ownership", ["eie aksjer", "eierskap", "aksjer i andre", "eie og forvalte",
                         "kjøp og salg av aksjer", "investere i aksjer"]),
    ("investment", ["investering", "kapitalforvaltning", "forvaltning av kapital",
                    "plassering av kapital", "verdipapir"]),
    ("real estate", ["fast eiendom", "eiendom", "utleie av lokaler"]),
    ("consulting", ["konsulent", "rådgivning", "rådgiving"]),
    ("trade", ["handel", "kjøp og salg av varer", "import", "eksport"]),
]

hit_cols = []
for label, terms in KEYWORDS:
    col = "kw_" + label.replace(" ", "_")
    cond = F.lit(False)
    for t in terms:
        cond = cond | F.col("formaal").contains(t)
    base = base.withColumn(col, cond)
    hit_cols.append((label, col))

# Priority order is the KEYWORDS order: an entity matching both "holding" and
# "real estate" is counted as holding. Stated so the numbers can be read.
bucket = F.when(F.length(F.col("formaal")) == 0, F.lit("no purpose text"))
for label, col in hit_cols:
    bucket = bucket.when(F.col(col), F.lit(label))
base = base.withColumn("purpose_bucket", bucket.otherwise(F.lit("other"))).cache()


def purpose_profile(name, condition):
    grp = base.filter(condition)
    n = grp.count()
    if not n:
        print("\n%s: no rows" % name)
        return
    print("\n%s (n = %d)" % (name, n))
    print("  Independent keyword hits (an entity may match several):")
    agg = grp.select(*[F.sum(F.when(F.col(col), 1).otherwise(0)).alias(col)
                       for _, col in hit_cols]).collect()[0]
    for label, col in hit_cols:
        print("    %-18s %8d  %6.2f%%" % (label, agg[col], agg[col] / n * 100))
    print("  Assigned bucket (priority order as listed above):")
    for _, r in (grp.groupBy("purpose_bucket").count()
                 .orderBy(F.desc("count")).toPandas().iterrows()):
        print("    %-18s %8d  %6.2f%%"
              % (r["purpose_bucket"], r["count"], r["count"] / n * 100))


print("\n\nPART B. Statutory purpose text")
print("=" * 72)

# Division 64 and 68 are the reference classes: entities the register itself
# calls financial or real estate. If the keywords do not separate those two,
# they are not trustworthy on division 00 either.
purpose_profile("Division 64 AS (register says financial) - keyword validation",
                F.col("div") == 64)
purpose_profile("Division 68 AS (register says real estate) - keyword validation",
                F.col("div") == 68)
purpose_profile("Division 00 AS, settled cohort",
                (F.col("div") == 0) & (F.col("cohort") == "settled"))
purpose_profile("Division 00 AS, recent cohort",
                (F.col("div") == 0) & (F.col("cohort") == "recent"))
purpose_profile("All other AS, settled cohort - baseline",
                (F.col("div") != 0) & (F.col("cohort") == "settled"))

# The most common purpose texts verbatim, which no keyword list anticipates well.
print("\nMost common purpose texts among division 00 AS (settled cohort):")
top = (base.filter((F.col("div") == 0) & (F.col("cohort") == "settled")
                   & (F.length(F.col("formaal")) > 0))
       .groupBy("formaal").count().orderBy(F.desc("count")).limit(20).toPandas())
for _, r in top.iterrows():
    text = r["formaal"][:90] + ("..." if len(r["formaal"]) > 90 else "")
    print("  %6d  %s" % (r["count"], text))

base.unpersist()
spark.stop()

Cohorts (AS with a naeringskode1)
  settled = 2019-01 to 2023-12     recent = 2026-03 to 2026-08
  recent   other       10809
  recent   div 00       5707
  settled  other      100479
  settled  div 00      24943

PART A. Profile by cohort and group (%% of that group)
Cohort   Group           n          VAT registered           Has employees   Has a reported filing      Flagged in a group       Under liquidation    Purpose text present
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
settled  div 00      24943                    7.50                    0.04                   99.94                    1.05                    1.33                  100.00
settled  other      100479                   57.07                   13.40                   99.86                    5.88                    1.76                  100.00
recent   div 00       5707                    1

In [9]:
import os

import pandas as pd
from pyspark.sql import SparkSession, functions as F

# ============================================================
# Company registrations by NACE section and year, as percentages.
# Four denominators, so the effect of each non-section bucket is
# visible rather than assumed.
# Reads the Parquet export of `companies`: 2 columns out of 63.
# ============================================================
YEAR_FROM = 1996
YEAR_TO = 2025

# Which date defines the cohort.
#   "registreringsdatoEnhetsregisteret" - entry into Enhetsregisteret. The
#       register opened in 1995, so years up to and including 1995 are a
#       backlog load of already-existing entities, not new formations.
#   "stiftelsesdato"                    - legal founding date. Can predate
#       registration by decades and is absent for many legal forms.
DATE_FIELD = "registreringsdatoEnhetsregisteret"

LEGAL_FORMS = None          # None = whole register; {"AS"} = aksjeselskap only

DATA_DIR = "/home/jovyan/data"
COMPANIES_PARQUET = os.path.join(DATA_DIR, "parquet", "companies")
OUT_PREFIX = os.path.join(DATA_DIR, "sector_by_year")

# ------------------------------------------------------------
# SN2025 division -> section, the standard in force in Enhetsregisteret
# since 2025-09-01. SN2025 is the Norwegian implementation of NACE Rev. 2.1.
# 87 divisions across 22 sections. Division 45 does not exist (its activities
# went to 46, 47 and 95); division 98 does.
# Section letters verified against SSB KLASS classification 6, codes valid
# 2026-01-01. English names are labels only and affect no number.
# ------------------------------------------------------------
SECTION_RANGES = [
    ("A",  1,  3, "Agriculture, forestry and fishing"),
    ("B",  5,  9, "Mining and quarrying"),
    ("C", 10, 33, "Manufacturing"),
    ("D", 35, 35, "Electricity, gas, steam, air conditioning"),
    ("E", 36, 39, "Water supply, sewerage, waste management"),
    ("F", 41, 43, "Construction"),
    ("G", 46, 47, "Wholesale and retail trade"),
    ("H", 49, 53, "Transportation and storage"),
    ("I", 55, 56, "Accommodation and food service"),
    ("J", 58, 60, "Publishing, broadcasting, content production"),
    ("K", 61, 63, "Telecoms, IT and information services"),
    ("L", 64, 66, "Financial and insurance activities"),
    ("M", 68, 68, "Real estate activities"),
    ("N", 69, 75, "Professional, scientific and technical"),
    ("O", 77, 82, "Administrative and support service"),
    ("P", 84, 84, "Public administration and defence"),
    ("Q", 85, 85, "Education"),
    ("R", 86, 88, "Human health and social work"),
    ("S", 90, 93, "Arts, entertainment and recreation"),
    ("T", 94, 96, "Other service activities, incl. vehicle repair"),
    ("U", 97, 98, "Households as employers"),
    ("V", 99, 99, "Extraterritorial organisations"),
]
SECTIONS = [letter for letter, _, _, _ in SECTION_RANGES]

UNSPEC = "00"           # 00.000 Uoppgitt - holding and investment entities
NO_CODE = "--"          # naeringskode1 absent entirely
UNMAPPED = "??"         # code string present but placed in no section
YEARS = range(YEAR_FROM, YEAR_TO + 1)

# Division 45 exists only in SN2007. Any hit means this mapping is the wrong
# vintage for the data and every letter from J onwards is suspect.
SN2007_ONLY_DIVISION = 45


def section_expr(code, division):
    """
    Map a NACE code to a section letter.

    Order matters:
      NO_CODE   naeringskode1.kode is null - the register says nothing
      UNMAPPED  the string is present but its division will not parse
      UNSPEC    division 00 - a deliberate register value, not an error
      A-V       a real SN2025 section
      UNMAPPED  anything else, including SN2007 leftovers like division 45
    """
    expr = F.lit(UNMAPPED)
    for letter, lo, hi, _ in reversed(SECTION_RANGES):
        expr = F.when(division.between(lo, hi), F.lit(letter)).otherwise(expr)
    return (F.when(code.isNull(), F.lit(NO_CODE))
             .when(division.isNull(), F.lit(UNMAPPED))
             .when(division == 0, F.lit(UNSPEC))
             .otherwise(expr))


spark = SparkSession.builder.appName("group13_sector_by_year").getOrCreate()

df = spark.read.parquet(COMPANIES_PARQUET)
if LEGAL_FORMS is not None:
    df = df.filter(F.col("organisasjonsform.kode").isin(list(LEGAL_FORMS)))

# Dates are strings, "YYYY-MM-DD". try_cast (Spark 4) yields null on a
# malformed value instead of failing the job under ANSI mode.
year = F.substring(F.col(DATE_FIELD), 1, 4).try_cast("int")

# naeringskode1.kode looks like "68.200"; the division is the first two digits.
code = F.col("naeringskode1.kode")
division = F.substring(code, 1, 2).try_cast("int")

work = df.select(
    year.alias("year"),
    division.alias("division"),
    section_expr(code, division).alias("section"),
    code.alias("nace"),
).cache()

# ------------------------------------------------------------
# Vintage guard, before anything is tabulated
# ------------------------------------------------------------
sn2007_hits = work.filter(F.col("division") == SN2007_ONLY_DIVISION).count()
if sn2007_hits:
    print("*" * 72)
    print("WARNING: %d entities carry division %02d, which exists only in SN2007."
          % (sn2007_hits, SN2007_ONLY_DIVISION))
    print("The section letters below assume SN2025. Every letter from J onwards")
    print("is wrong for those rows. Stop and resolve the vintage before using")
    print("these tables.")
    print("*" * 72)
else:
    print("Vintage guard: no division %02d present, consistent with SN2025.\n"
          % SN2007_ONLY_DIVISION)

# ------------------------------------------------------------
# Coverage, measured rather than assumed
# ------------------------------------------------------------
total_rows = work.count()
no_year = work.filter(F.col("year").isNull()).count()
in_window = work.filter(F.col("year").between(YEAR_FROM, YEAR_TO))
window_rows = in_window.count()

print("Source:            %s" % COMPANIES_PARQUET)
print("Cohort date field: %s" % DATE_FIELD)
print("Legal forms:       %s" % (sorted(LEGAL_FORMS) if LEGAL_FORMS else "ALL"))
print()
print("Rows in register:        %9d" % total_rows)
print("  no date value:         %9d" % no_year)
print("  outside %d-%d:      %9d" % (YEAR_FROM, YEAR_TO,
                                     total_rows - window_rows - no_year))
print("  in window, tabulated:  %9d" % window_rows)

# The years around the window start, so the size of the excluded mass is
# visible rather than asserted. Enhetsregisteret's opening year should dominate.
edge = (work.filter(F.col("year").between(YEAR_FROM - 4, YEAR_FROM))
        .groupBy("year").count().orderBy("year").toPandas())
print("\nRegistrations just before and at the window start:")
for _, r in edge.iterrows():
    print("  %d  %9d" % (r["year"], r["count"]))

# What landed in the unmapped bucket. A null `nace` means the string was
# present but unparseable; anything else is a code outside SN2025.
unmapped = (in_window.filter(F.col("section") == UNMAPPED)
            .groupBy("nace").count().orderBy(F.desc("count")).limit(10).toPandas())
if len(unmapped):
    print("\nCodes placed in '%s' (top 10):" % UNMAPPED)
    for _, r in unmapped.iterrows():
        label = "(unparseable)" if pd.isna(r["nace"]) else r["nace"]
        print("  %-14s %9d" % (label, r["count"]))
else:
    print("\nEvery naeringskode1 present maps to an SN2025 section or to '%s'." % UNSPEC)

# ------------------------------------------------------------
# The aggregation: 1.17M rows down to at most 30 x 25 cells
# ------------------------------------------------------------
counts = in_window.groupBy("year", "section").count().toPandas()

work.unpersist()
spark.stop()

# ------------------------------------------------------------
# Pivot once, then renormalise per denominator
# ------------------------------------------------------------
ALL_COLUMNS = SECTIONS + [UNSPEC, NO_CODE, UNMAPPED]
table = (counts.pivot(index="year", columns="section", values="count")
         .reindex(index=YEARS, columns=ALL_COLUMNS)
         .fillna(0).astype(int))

print("\nSN2025 sections")
for letter, lo, hi, desc in SECTION_RANGES:
    span = "%02d" % lo if lo == hi else "%02d-%02d" % (lo, hi)
    print("  %s  %-6s %s" % (letter, span, desc))
print("  %s      holding and investment entities (see note below)" % UNSPEC)
print("  %s      naeringskode1 absent" % NO_CODE)
print("  %s      code present, placed in no section" % UNMAPPED)

print("""
Note on '00'. Brreg assigns 00.000 Uoppgitt to newly registered investment and
holding entities. It is a valid register code, not missing data, and Brreg
states it may take 2-3 years before a more precise code is assigned
(https://www.brreg.no/bedrift/naeringskoder/, retrieved 2026-09-09). It is not
a division in SN2007 or SN2025 as published in SSB KLASS, so it is kept as its
own column rather than folded into any section. Two consequences for the tables:
  - Section L (financial) understates holding activity from 2018 onwards, when
    entities that would previously have been coded 64 began arriving as 00.
  - The '00' share per registration year mixes two processes that a single
    snapshot cannot separate: the rate at which 00 is assigned at registration,
    and the rate at which it is later replaced. Treat its trend with caution.
""")

# Each variant selects its own columns; the denominator is the row sum of
# exactly those columns, so every table's rows sum to 100.
VARIANTS = [
    ("pct_all", SECTIONS + [UNSPEC, NO_CODE, UNMAPPED],
     "TABLE 1  All registrations (every bucket included)"),
    ("pct_sections", SECTIONS,
     "TABLE 2  Sections only (all buckets excluded)"),
    ("pct_nocode", SECTIONS + [NO_CODE],
     "TABLE 3  Sections plus '%s' (no code), excluding '%s' and '%s'"
     % (NO_CODE, UNSPEC, UNMAPPED)),
    ("pct_unspec", SECTIONS + [UNSPEC],
     "TABLE 4  Sections plus '%s' (holding/investment), excluding '%s' and '%s'"
     % (UNSPEC, NO_CODE, UNMAPPED)),
]

for suffix, cols, title in VARIANTS:
    subset = table[cols]
    totals = subset.sum(axis=1)
    # replace(0, NA) so a year with no rows gives NaN, not a divide error
    pct = subset.div(totals.replace(0, pd.NA), axis=0) * 100

    print("\n" + title)
    header = "Year " + "".join("%7s" % c for c in cols) + "%12s" % "Total"
    print(header)
    print("-" * len(header))
    for y in YEARS:
        if totals[y] == 0:
            print("%-5d%s%12d" % (y, "".join("%7s" % "-" for _ in cols), 0))
            continue
        cells = "".join("%7.2f" % pct.loc[y, c] for c in cols)
        print("%-5d%s%12d" % (y, cells, totals[y]))

    # The total column is named "total" rather than "N", which would collide
    # with NACE section N.
    out = pct.round(2)
    out["total"] = totals
    path = "%s_%s.csv" % (OUT_PREFIX, suffix)
    out.to_csv(path, index_label="year")
    print("Written: %s" % path)

Vintage guard: no division 45 present, consistent with SN2025.

Source:            /home/jovyan/data/parquet/companies
Cohort date field: registreringsdatoEnhetsregisteret
Legal forms:       ALL

Rows in register:          1171373
  no date value:                 0
  outside 1996-2025:         152799
  in window, tabulated:    1018574

Registrations just before and at the window start:
  1995      91047
  1996      12095

Every naeringskode1 present maps to an SN2025 section or to '00'.

SN2025 sections
  A  01-03  Agriculture, forestry and fishing
  B  05-09  Mining and quarrying
  C  10-33  Manufacturing
  D  35     Electricity, gas, steam, air conditioning
  E  36-39  Water supply, sewerage, waste management
  F  41-43  Construction
  G  46-47  Wholesale and retail trade
  H  49-53  Transportation and storage
  I  55-56  Accommodation and food service
  J  58-60  Publishing, broadcasting, content production
  K  61-63  Telecoms, IT and information services
  L  64-66  Financial and 

In [4]:
import requests
from collections import Counter

import pymongo

# ============================================================
# Which classification vintage does naeringskode1 use?
# Compares the register's actual divisions against SN2007 and
# SN2025 as published by SSB KLASS (classification 6).
# ============================================================
KLASS_URL = "https://data.ssb.no/api/klass/v1/classifications/6/codesAt"
DATE_SN2007 = "2024-01-01"      # before the 2025-09-01 switch
DATE_SN2025 = "2026-01-01"      # after it

MONGO_URI = "mongodb://mongodb:27017/"


def codes_at(date):
    """All codes valid on a date. `date` is the only range parameter codesAt takes."""
    r = requests.get(KLASS_URL, params={"date": date},
                     headers={"Accept": "application/json"}, timeout=30)
    if not r.ok:
        # KLASS returns a problem+json body explaining the rejection
        print("HTTP %s for %s\n%s" % (r.status_code, r.url, r.text[:400]))
        r.raise_for_status()
    return r.json()["codes"]


def divisions(codes):
    """
    Division -> section letter.
    Levels are derived, not assumed: the division level is the one whose codes
    are two digits and whose parentCode is a single letter.
    """
    by_level = {}
    for c in codes:
        by_level.setdefault(str(c["level"]), []).append(c)

    for level in sorted(by_level):
        sample = by_level[level][0]
        print("  level %s: %4d codes   e.g. code=%r parentCode=%r  %s"
              % (level, len(by_level[level]), sample["code"],
                 sample["parentCode"], sample["name"][:40]))

    div_level = None
    for level, items in by_level.items():
        first = items[0]
        if (len(first["code"]) == 2 and first["code"].isdigit()
                and first["parentCode"] and len(first["parentCode"]) == 1):
            div_level = level
            break
    if div_level is None:
        raise RuntimeError("Could not identify the division level - inspect the output above")

    print("  -> using level %s as the division level" % div_level)
    return {c["code"]: c["parentCode"] for c in by_level[div_level]}


print("SN2007 (codes valid %s):" % DATE_SN2007)
sn2007 = divisions(codes_at(DATE_SN2007))
print("\nSN2025 (codes valid %s):" % DATE_SN2025)
sn2025 = divisions(codes_at(DATE_SN2025))

print("\nSN2007 divisions: %d    SN2025 divisions: %d" % (len(sn2007), len(sn2025)))

# Divisions that exist in only one vintage - the tell-tales.
print("Only in SN2007: %s" % (sorted(set(sn2007) - set(sn2025)) or "none"))
print("Only in SN2025: %s" % (sorted(set(sn2025) - set(sn2007)) or "none"))

# Same division number, different section letter - the silent failure mode.
shifted = sorted(d for d in set(sn2007) & set(sn2025) if sn2007[d] != sn2025[d])
print("\nSame division, different section: %d divisions" % len(shifted))
for d in shifted:
    print("  %s  SN2007=%s  SN2025=%s" % (d, sn2007[d], sn2025[d]))

# ------------------------------------------------------------
# What the register actually holds
# ------------------------------------------------------------
client = pymongo.MongoClient(MONGO_URI)
companies = client["companiesdb"]["companies"]

pipeline = [
    {"$match": {"naeringskode1.kode": {"$exists": True}}},
    {"$group": {"_id": {"$substrBytes": ["$naeringskode1.kode", 0, 2]},
                "n": {"$sum": 1}}},
]
observed = {r["_id"]: r["n"] for r in companies.aggregate(pipeline)}
print("\nDistinct divisions in companies: %d" % len(observed))

buckets = Counter()
for div, n in observed.items():
    in07, in25 = div in sn2007, div in sn2025
    key = ("both" if in07 and in25 else
           "SN2007 only" if in07 else
           "SN2025 only" if in25 else "neither")
    buckets[key] += n

total = sum(buckets.values())
print("\nEntities by vintage evidence:")
for key, n in buckets.most_common():
    print("  %-12s %9d  (%.2f%%)" % (key, n, n / total * 100))

orphans = sorted((d, n) for d, n in observed.items()
                 if d not in sn2007 and d not in sn2025)
print("\nDivisions in neither vintage: %d" % len(orphans))
for d, n in orphans[:20]:
    print("  %s  %9d" % (d, n))

SN2007 (codes valid 2024-01-01):
  level 1:   21 codes   e.g. code='A' parentCode=None  Jordbruk, skogbruk og fiske
  level 2:   87 codes   e.g. code='01' parentCode='A'  Jordbruk og tjenester tilknyttet jordbru
  level 3:  270 codes   e.g. code='01.1' parentCode='01'  Dyrking av ettårige vekster
  level 4:  613 codes   e.g. code='01.11' parentCode='01.1'  Dyrking av korn (unntatt ris), belgvekst
  level 5:  820 codes   e.g. code='01.110' parentCode='01.11'  Dyrking av korn (unntatt ris), belgvekst
  -> using level 2 as the division level

SN2025 (codes valid 2026-01-01):
  level 1:   22 codes   e.g. code='A' parentCode=None  Jordbruk, skogbruk og fiske
  level 2:   87 codes   e.g. code='01' parentCode='A'  Jordbruk og tjenester tilknyttet jordbru
  level 3:  287 codes   e.g. code='01.1' parentCode='01'  Dyrking av ettårige vekster
  level 4:  651 codes   e.g. code='01.11' parentCode='01.1'  Dyrking av korn, unntatt ris, belgvekste
  level 5:  738 codes   e.g. code='01.110' parentCode=